# FinBERT Finetuning (sentiment_dataset.csv)

Este notebook realiza **finetuning normal** (entrenando todos los parámetros) de `ProsusAI/finbert` para clasificación de sentimiento financiero con 3 clases: **negativo, neutral, positivo**.

In [ ]:
# Instalación de dependencias requeridas
%pip install -q transformers torch datasets pandas scikit-learn

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

SEED = 42
MODEL_NAME = "ProsusAI/finbert"
LABEL2ID = {"negative": 0, "neutral": 1, "positive": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [ ]:
# Carga del dataset desde data/sentiment_dataset.csv
DATA_PATH = os.path.abspath(os.path.join(os.getcwd(), "..", "data", "sentiment_dataset.csv"))
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
# Limpieza y preparación de datos
required_cols = ["text", "sentiment", "source"]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Faltan columnas requeridas: {missing_cols}")

df = df[required_cols].copy()
df["text"] = df["text"].astype(str).str.strip()
df["sentiment"] = df["sentiment"].astype(str).str.strip().str.lower()

df = df.dropna(subset=["text", "sentiment"])
df = df[df["text"] != ""]
df = df[df["sentiment"].isin(LABEL2ID.keys())]
df = df.drop_duplicates(subset=["text", "sentiment"]).reset_index(drop=True)

# Mapeo de sentimientos a índices numéricos
df["label"] = df["sentiment"].map(LABEL2ID).astype(int)

print(f"Total de muestras limpias: {len(df):,}")
print(df["sentiment"].value_counts())
df.head()

In [ ]:
# Split estratificado: train 80%, valid 10%, test 10%
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df["label"],
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=SEED,
    stratify=temp_df["label"],
)

print("Shapes:", train_df.shape, val_df.shape, test_df.shape)
print("Ratios:", len(train_df)/len(df), len(val_df)/len(df), len(test_df)/len(df))

In [ ]:
# Creación de datasets compatibles con HuggingFace
train_ds = Dataset.from_pandas(train_df[["text", "label"]].reset_index(drop=True))
val_ds = Dataset.from_pandas(val_df[["text", "label"]].reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df[["text", "label"]].reset_index(drop=True))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

train_tok = train_ds.map(tokenize_batch, batched=True)
val_tok = val_ds.map(tokenize_batch, batched=True)
test_tok = test_ds.map(tokenize_batch, batched=True)

cols = ["input_ids", "attention_mask", "label"]
if "token_type_ids" in train_tok.column_names:
    cols.append("token_type_ids")

train_tok.set_format(type="torch", columns=cols)
val_tok.set_format(type="torch", columns=cols)
test_tok.set_format(type="torch", columns=cols)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
# Configuración del modelo FinBERT (3 labels => 3 logits)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

print("Número de labels del modelo:", model.config.num_labels)
print("id2label:", model.config.id2label)

In [ ]:
# Métricas: accuracy, precision, recall, F1
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

In [ ]:
# Entrenamiento con Trainer de HuggingFace
training_args = TrainingArguments(
    output_dir="../models/finbert_finetuned",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=1,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
train_result

In [ ]:
# Evaluación en validación y test
val_metrics = trainer.evaluate(eval_dataset=val_tok, metric_key_prefix="val")
test_metrics = trainer.evaluate(eval_dataset=test_tok, metric_key_prefix="test")

print("Validation metrics:")
print(val_metrics)
print("\nTest metrics:")
print(test_metrics)

In [ ]:
# Función de inferencia: devuelve logits y probabilidades para nuevos textos
def predict_texts(texts, model, tokenizer, max_length=128):
    if isinstance(texts, str):
        texts = [texts]

    device = model.device
    enc = tokenizer(
        texts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_length,
    )
    enc = {k: v.to(device) for k, v in enc.items()}

    model.eval()
    with torch.no_grad():
        out = model(**enc)
        logits = out.logits
        probs = torch.softmax(logits, dim=-1)

    logits_np = logits.cpu().numpy()
    probs_np = probs.cpu().numpy()
    pred_ids = probs_np.argmax(axis=-1)

    results = []
    for i, txt in enumerate(texts):
        results.append({
            "text": txt,
            "logits": logits_np[i].tolist(),
            "probabilities": probs_np[i].tolist(),
            "predicted_label_id": int(pred_ids[i]),
            "predicted_label": ID2LABEL[int(pred_ids[i])],
        })
    return results

In [ ]:
# Ejemplos de uso con predicciones
example_texts = [
    "Goldman Sachs stock price target raised to $367 from $358 at Oppenheimer",
    "Baltic Dry Index sheds 5%",
    "Ocean impact investment fund reaches $92m second close",
]

preds = predict_texts(example_texts, trainer.model, tokenizer)
for p in preds:
    print("-" * 100)
    print("Text:", p["text"])
    print("Logits [neg, neu, pos]:", np.round(p["logits"], 4))
    print("Probabilidades [neg, neu, pos]:", np.round(p["probabilities"], 4))
    print("Predicción:", p["predicted_label"])

In [ ]:
# Guardado del modelo finetuneado
SAVE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "models", "finbert_finetuned"))
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Modelo y tokenizer guardados en: {SAVE_DIR}")